# Syntax-vs-stenosis overlap, split by split (follow-up to the first duplicate check)

**No training, no GPU. About 3 to 5 minutes.**

The first check compared the whole syntax folder (1500 images) with the stenosis images and found 123 syntax images with a 32x32 correlation of at least 0.98 to a stenosis test image. The thesis vessel head used only the **syntax train** images (1000). This notebook splits the answer by syntax split (train / val / test) and by stenosis split, and also counts exact copies inside the 1500 stenosis images.

**Input:** ARCADE dataset (nirmalgaud). **Output:** `dup2_outputs.zip` - send it back.

In [1]:
import os, json, hashlib, zipfile, warnings
from pathlib import Path
import numpy as np, pandas as pd, cv2
warnings.filterwarnings("ignore")
ON_KAGGLE = Path("/kaggle/working").exists()
BASE = Path("/kaggle/working") if ON_KAGGLE else (Path("/content") if Path("/content").exists() else Path.cwd())
OUT = BASE / "dup2_outputs"; OUT.mkdir(parents=True, exist_ok=True)
def find_root():
    for root in [Path(p) for p in os.environ.get("UNI_DATA_ROOTS", "/kaggle/input").split(":")] + [Path.home() / ".cache" / "kagglehub"]:
        if root.exists():
            c = [p for p in root.rglob("train.json") if "stenosis" in str(p).lower()]
            if c: return c[0].parent.parent.parent
    raise RuntimeError("ARCADE dataset not found.")
STEN = find_root(); SYN = STEN.parent / "syntax"; print("stenosis:", STEN, "| syntax:", SYN, SYN.exists())
assert SYN.exists(), "syntax folder not found next to the stenosis folder"

def load(root, split):
    coco = json.load(open(root / split / "annotations" / f"{split}.json"))
    ims = sorted(coco["images"], key=lambda d: d["id"])
    arr = [cv2.imread(str(root / split / "images" / im["file_name"]), cv2.IMREAD_GRAYSCALE) for im in ims]
    return arr, [f"{root.name}/{split}/{im['file_name']}" for im in ims]
D = {}
for root in (STEN, SYN):
    for sp in ("train", "val", "test"):
        D[(root.name, sp)] = load(root, sp); print(root.name, sp, len(D[(root.name, sp)][0]))

def z32(imgs):
    a = np.stack([cv2.resize(g, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32).ravel() for g in imgs])
    a -= a.mean(1, keepdims=True); return a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)
def g128(g): return cv2.resize(g, (128, 128), interpolation=cv2.INTER_AREA).astype(np.float32)
def ssim(a, b):
    C1, C2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2; mu = lambda x: cv2.GaussianBlur(x, (11, 11), 1.5)
    ma, mb = mu(a), mu(b); saa, sbb, sab = mu(a * a) - ma ** 2, mu(b * b) - mb ** 2, mu(a * b) - ma * mb
    return float((((2 * ma * mb + C1) * (2 * sab + C2)) / ((ma ** 2 + mb ** 2 + C1) * (saa + sbb + C2))).mean())
Z = {k: z32(v[0]) for k, v in D.items()}
H = {k: [hashlib.md5(g.tobytes()).hexdigest() for g in v[0]] for k, v in D.items()}

rows, detail = [], []
for ss in ("train", "val", "test"):                       # syntax split
    for ts in ("train", "val", "test"):                   # stenosis split
        C = Z[("syntax", ss)] @ Z[("stenosis", ts)].T
        mx = C.max(1)
        hs = set(H[("stenosis", ts)])
        exact = sum(h in hs for h in H[("syntax", ss)])
        close = np.where(mx >= 0.98)[0]
        sims = []
        for i in close:
            j = int(C[i].argmax()); sims.append(ssim(g128(D[("syntax", ss)][0][i]), g128(D[("stenosis", ts)][0][j])))
            detail.append(dict(syntax=D[("syntax", ss)][1][i], stenosis=D[("stenosis", ts)][1][j], corr32=round(float(mx[i]), 4), ssim128=round(sims[-1], 4)))
        rows.append(dict(syntax_split=ss, stenosis_split=ts, n_syntax=len(mx), max_corr32=round(float(mx.max()), 4), n_corr_ge_098=len(close),
                         n_corr_ge_098_and_ssim_ge_09=int(sum(s >= 0.9 for s in sims)), n_exact_copies=int(exact)))
T = pd.DataFrame(rows); T.to_csv(OUT / "syntax_split_vs_stenosis_split.csv", index=False); print(T.to_string(index=False))
pd.DataFrame(detail).to_csv(OUT / "syntax_stenosis_close_pairs.csv", index=False)

# exact duplicates inside the stenosis set (pool and test), counted by pair
allH = [(h, n) for k in [("stenosis", s) for s in ("train", "val", "test")] for h, n in zip(H[k], D[k][1])]
ser = pd.Series([h for h, _ in allH]); dup = ser[ser.duplicated(keep=False)]
print("\nexact-duplicate groups inside the 1500 stenosis images:", dup.nunique(), "| images involved:", len(dup))
g = {}
for h, n in allH: g.setdefault(h, []).append(n)
pd.DataFrame([dict(files=";".join(v)) for v in g.values() if len(v) > 1]).to_csv(OUT / "exact_duplicates_inside_stenosis.csv", index=False)
print("pairs between stenosis test and stenosis train/val (exact):", sum(any("/test/" in x for x in v) and any(("/train/" in x or "/val/" in x) for x in v) for v in g.values() if len(v) > 1))
z = BASE / "dup2_outputs.zip"
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in OUT.iterdir(): zf.write(f, f.name)
print("\nDONE. Send back:", z)

stenosis: /kaggle/input/datasets/nirmalgaud/arcade-dataset/arcade/stenosis | syntax: /kaggle/input/datasets/nirmalgaud/arcade-dataset/arcade/syntax True
stenosis train 1000
stenosis val 200
stenosis test 300
syntax train 1000
syntax val 200
syntax test 300
syntax_split stenosis_split  n_syntax  max_corr32  n_corr_ge_098  n_corr_ge_098_and_ssim_ge_09  n_exact_copies
       train          train      1000      1.0000            250                           247               0
       train            val      1000      1.0000             21                            18               0
       train           test      1000      0.9608              0                             0               0
         val          train       200      1.0000             73                            71               0
         val            val       200      0.9999              1                             1               0
         val           test       200      0.9670              0             